<a href="https://colab.research.google.com/github/Kamalll7/Final_Project_PCD-Kelompok_6/blob/main/notebook/02_Number_Plate_Detection_ROI_HOG_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## 1. Pendahuluan

### 1.1 Latar Belakang Singkat

Undang-Undang Nomor 22 Tahun 2009 mewajibkan kendaraan bermotor yang beroperasi di jalan
dilengkapi **Tanda Nomor Kendaraan Bermotor (TNKB)**. Deteksi keberadaan plat nomor merupakan
langkah awal yang krusial dalam sistem *Automatic License Plate Recognition* (ALPR), sebelum
tahap segmentasi dan pengenalan karakter.

### 1.2 Rumusan Masalah yang Dijawab Notebook Ini

> **Bagaimana mendeteksi keberadaan plat nomor pada sepeda motor menggunakan metode
> pengolahan citra digital dan machine learning?** (Rumusan Masalah #2)

### 1.3 Tujuan

Mengembangkan dan mengevaluasi metode deteksi keberadaan plat nomor sepeda motor menggunakan
**RoI** untuk mempersempit area pencarian, **HOG** untuk mengekstraksi deskriptor tepi/gradien,
dan **SVM** untuk klasifikasi biner.

### 1.4 Mengapa Deteksi Plat Motor Lebih Sulit daripada Plat Mobil?

Proposal §2.6 mencatat beberapa tantangan spesifik:

1. **Latar belakang kompleks** — plat motor bercampur dengan bodi kendaraan, knalpot, rantai,
   dan elemen lain di sekitarnya, berbeda dengan plat mobil yang relatif bersih.
2. **Ukuran kecil dan orientasi bervariasi** — plat motor lebih kecil dan dapat miring
   (*yaw*, *pitch*, *roll*).
3. **Literatur lebih jarang** — deteksi plat motor jauh lebih sedikit diteliti dibanding plat mobil.

### 1.5 Strategi: RoI Dulu, Baru Cari Plat

Hsu et al. (2015) — dirujuk proposal §2.6 — membandingkan dua skema:

| Skema | Cara kerja | Hasil |
|---|---|---|
| **Deteksi motor dulu**, lalu cari plat di dalamnya | meniru cara kerja visual manusia | **false positive jauh lebih sedikit**, deteksi lebih cepat |
| Deteksi plat langsung pada seluruh citra | seperti *sliding window* pada piramida citra | lebih banyak false positive, lebih lambat |

Contoh angka dari studi tersebut: deteksi motor **279 ms**, lalu plat hanya butuh **~8 ms**
tambahan di dalam area motor — jauh lebih cepat dibanding memindai seluruh citra.

**Karena itu notebook ini mengadopsi skema pertama**: RoI ditentukan lebih dulu (dari anotasi
`rider`), lalu pencarian plat dilakukan **hanya di dalam RoI tersebut**.

---
## 2. Setup Environment

Dirancang untuk **Google Colab** (sesuai §3.10 proposal) maupun eksekusi lokal.

**Perangkat lunak** (§3.10): `Python`, `OpenCV`, `scikit-image` (HOG), `scikit-learn` (SVM & metrik),
`NumPy`, `Matplotlib`, `Pandas`.

In [ ]:
import sys, subprocess

REQUIRED = ["opencv-python-headless", "scikit-image", "scikit-learn",
            "matplotlib", "pandas", "numpy", "pillow"]

def _ensure(pkgs):
    missing = []
    for p in pkgs:
        mod = {"opencv-python-headless": "cv2", "scikit-image": "skimage",
               "scikit-learn": "sklearn", "pillow": "PIL"}.get(p, p)
        try:
            __import__(mod)
        except ImportError:
            missing.append(p)
    if missing:
        print("Memasang:", missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)

_ensure(REQUIRED)
print("Dependensi siap.")

In [ ]:
import os, json, random, warnings, itertools
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, cohen_kappa_score,
                             roc_curve, auc, classification_report,
                             precision_recall_curve, average_precision_score)

warnings.filterwarnings("ignore")

# ---- Reproducibility -------------------------------------------------------
SEED = 7
random.seed(SEED)
np.random.seed(SEED)

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 110,
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
})

import sklearn, skimage
print("NumPy       :", np.__version__)
print("OpenCV      :", cv2.__version__)
print("scikit-learn:", sklearn.__version__)
print("scikit-image:", skimage.__version__)

---
## 3. Akuisisi Dataset

Dataset yang sama dengan Jalur 1, dengan struktur anotasi **YOLO**:

```
dataset/
├── train/{images,labels}/
└── val/{images,labels}/
```

**Kelas:**

| ID | Nama kelas |
|---|---|
| 0 | `with helmet` |
| 1 | `without helmet` |
| 2 | `rider` |
| 3 | `number plate` |

Kelas yang dipakai notebook ini: **`rider`** (untuk menentukan RoI) dan
**`number plate`** (sebagai *ground truth*).

In [ ]:
DATASET_SLUG = "aneesarom/rider-with-helmet-without-helmet-number-plate"
CLASS_NAMES  = ["with helmet", "without helmet", "rider", "number plate"]

def _looks_like_dataset(p: Path) -> bool:
    return (p / "train" / "images").is_dir() and (p / "train" / "labels").is_dir()

def resolve_dataset() -> Path:
    candidates = []
    env = os.environ.get("PCD_DATASET_DIR")
    if env:
        candidates.append(Path(env))
    candidates += [Path("./dataset"), Path("../dataset"), Path("/content/dataset"),
                   Path.home() / ".cache/kagglehub/datasets/aneesarom/rider-with-helmet-without-helmet-number-plate"]
    for base in candidates:
        if not base.exists():
            continue
        if _looks_like_dataset(base):
            return base
        for sub in base.rglob("train"):
            if (sub / "images").is_dir() and (sub / "labels").is_dir():
                return sub.parent
    print("Dataset belum ditemukan secara lokal, mengunduh via kagglehub ...")
    try:
        import kagglehub
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=False)
        import kagglehub
    root = Path(kagglehub.dataset_download(DATASET_SLUG))
    for sub in [root] + list(root.rglob("*")):
        if _looks_like_dataset(sub):
            return sub
    raise FileNotFoundError("Struktur dataset tidak dikenali setelah unduhan.")

DATA_ROOT = resolve_dataset()
print("Dataset root :", DATA_ROOT.resolve())
print("Train images :", len(list((DATA_ROOT / 'train' / 'images').glob('*'))))
print("Val images   :", len(list((DATA_ROOT / 'val' / 'images').glob('*'))))

### 3.1 Pembaca Citra yang Robust

Satu berkas berekstensi `.jpg` pada dataset sebenarnya berformat **AVIF**, sehingga `cv2.imread()`
mengembalikan `None`. Digunakan pembaca dengan *fallback* ke `Pillow` agar tidak ada data yang
hilang secara diam-diam.

In [ ]:
def imread_robust(path) -> np.ndarray:
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is not None:
        return img
    from PIL import Image
    with Image.open(path) as im:
        arr = np.array(im.convert("RGB"))
    return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)


def parse_label_file(path: Path):
    boxes = []
    for line in path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cls = int(parts[0])
        x, y, w, h = map(float, parts[1:5])
        boxes.append({"cls": cls,
                      "x1": x - w / 2, "y1": y - h / 2,
                      "x2": x + w / 2, "y2": y + h / 2})
    return boxes


def build_index(root: Path) -> pd.DataFrame:
    rows = []
    for split in ("train", "val"):
        for lbl in sorted((root / split / "labels").glob("*.txt")):
            stem = lbl.stem
            img_path = root / split / "images" / (stem + ".jpg")
            if not img_path.exists():
                hits = list((root / split / "images").glob(stem + ".*"))
                if not hits:
                    continue
                img_path = hits[0]
            boxes = parse_label_file(lbl)
            row = {"split": split, "stem": stem, "image_path": str(img_path),
                   "n_objects": len(boxes), "boxes": boxes}
            for cid, cname in enumerate(CLASS_NAMES):
                row["n_" + cname] = sum(1 for b in boxes if b["cls"] == cid)
            rows.append(row)
    return pd.DataFrame(rows)

df = build_index(DATA_ROOT)
print(f"Total citra terindeks: {len(df)}")

bad = [p for p in df["image_path"] if imread_robust(p) is None]
print("Citra gagal dibaca:", len(bad))

total = {c: int(df["n_" + c].sum()) for c in CLASS_NAMES}
print("\nJumlah bounding box per kelas:")
for k, v in total.items():
    print(f"  {k:16s} : {v}")

---
## 4. Eksplorasi Data: Geometri Plat Nomor

Karena jalur ini bertumpu pada **RoI**, analisis paling penting adalah **di mana plat nomor
berada relatif terhadap *bounding box* pengendara**. Analisis ini menjadi dasar kuantitatif
penentuan RoI pada §3.5 proposal.

Selain posisi, dianalisis pula **ukuran dan rasio aspek** plat — keduanya menentukan
ukuran *window* dan skala yang dipakai pada tahap pembentukan kandidat.

In [ ]:
def iou_norm(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

def box_tuple(b):
    return (b["x1"], b["y1"], b["x2"], b["y2"])

# Relasi plat <-> pengendara
plate_rel = []
for _, row in df.iterrows():
    riders = [b for b in row["boxes"] if b["cls"] == 2]
    plates = [b for b in row["boxes"] if b["cls"] == 3]
    for r in riders:
        rb = box_tuple(r)
        rw, rh = rb[2] - rb[0], rb[3] - rb[1]
        for p in plates:
            pb = box_tuple(p)
            if iou_norm(rb, pb) <= 0.001:
                continue
            plate_rel.append({
                "cx_rel": ((pb[0] + pb[2]) / 2 - rb[0]) / rw,
                "cy_rel": ((pb[1] + pb[3]) / 2 - rb[1]) / rh,
                "w_rel": (pb[2] - pb[0]) / rw,
                "h_rel": (pb[3] - pb[1]) / rh,
            })

pr_df = pd.DataFrame(plate_rel)
print("Pasangan plat-pengendara:", len(pr_df))
print()
for col in ["cx_rel", "cy_rel", "w_rel", "h_rel"]:
    q = pr_df[col].quantile([0.05, 0.25, 0.50, 0.75, 0.95])
    print(f"{col:8s}  p5={q.iloc[0]:.3f}  p25={q.iloc[1]:.3f}  "
          f"med={q.iloc[2]:.3f}  p75={q.iloc[3]:.3f}  p95={q.iloc[4]:.3f}")

In [ ]:
# Ukuran plat dalam piksel (perlu resolusi asli) dan rasio aspeknya
px_w, px_h, aspect = [], [], []
for _, row in df.iterrows():
    img = imread_robust(row["image_path"])
    if img is None:
        continue
    H, W = img.shape[:2]
    for b in row["boxes"]:
        if b["cls"] != 3:
            continue
        w = (b["x2"] - b["x1"]) * W
        h = (b["y2"] - b["y1"]) * H
        if w < 2 or h < 2:
            continue
        px_w.append(w); px_h.append(h); aspect.append(w / h)

px_w, px_h, aspect = map(np.array, (px_w, px_h, aspect))
print(f"Lebar plat (px)   : p10={np.percentile(px_w,10):.0f}  median={np.median(px_w):.0f}  "
      f"p90={np.percentile(px_w,90):.0f}")
print(f"Tinggi plat (px)  : p10={np.percentile(px_h,10):.0f}  median={np.median(px_h):.0f}  "
      f"p90={np.percentile(px_h,90):.0f}")
print(f"Rasio aspek w/h   : p10={np.percentile(aspect,10):.2f}  median={np.median(aspect):.2f}  "
      f"p90={np.percentile(aspect,90):.2f}")
print(f"\n-> Rasio aspek plat (median {np.median(aspect):.2f}) menjadi acuan AR_WINDOW pada §7")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))

axes[0].hist(pr_df["cy_rel"], bins=30, color="#457b9d", edgecolor="black", linewidth=0.4)
axes[0].axvline(0.45, color="crimson", ls="--", lw=1.3, label="batas atas RoI")
axes[0].axvline(0.92, color="crimson", ls="--", lw=1.3, label="batas bawah RoI")
axes[0].set_xlabel("Posisi vertikal plat / tinggi pengendara")
axes[0].set_ylabel("Frekuensi")
axes[0].set_title("Posisi vertikal plat")
axes[0].legend(fontsize=7)

axes[1].hist(aspect, bins=30, color="#e9c46a", edgecolor="black", linewidth=0.4)
axes[1].axvline(np.median(aspect), color="crimson", ls="--", lw=1.3,
                label=f"median={np.median(aspect):.2f}")
axes[1].set_xlabel("Rasio aspek plat (w/h)")
axes[1].set_ylabel("Frekuensi")
axes[1].set_title("Rasio aspek plat")
axes[1].legend(fontsize=7)

axes[2].scatter(pr_df["w_rel"], pr_df["h_rel"], s=16, alpha=0.65, color="#2a9d8f")
axes[2].set_xlabel("Lebar plat / lebar pengendara")
axes[2].set_ylabel("Tinggi plat / tinggi pengendara")
axes[2].set_title("Ukuran relatif plat")

plt.tight_layout()
plt.show()

print("Proporsi plat yang pusatnya berada di antara 45%%–92%% tinggi pengendara: %.1f%%"
      % (100 * pr_df["cy_rel"].between(0.45, 0.92).mean()))

In [ ]:
# Visualisasi contoh dengan anotasi plat
COLORS = {0: "#2a9d8f", 1: "#e76f51", 2: "#457b9d", 3: "#e9c46a"}

sample = df[df["n_number plate"] > 0].sample(6, random_state=SEED)
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    img = imread_robust(row["image_path"])
    H, W = img.shape[:2]
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    for b in row["boxes"]:
        x1, y1 = b["x1"] * W, b["y1"] * H
        w, h = (b["x2"] - b["x1"]) * W, (b["y2"] - b["y1"]) * H
        ax.add_patch(Rectangle((x1, y1), w, h, fill=False,
                               edgecolor=COLORS[b["cls"]],
                               lw=2.0 if b["cls"] == 3 else 1.2))
        ax.text(x1, max(y1 - 3, 8), CLASS_NAMES[b["cls"]], fontsize=6, color="white",
                bbox=dict(fc=COLORS[b["cls"]], ec="none", pad=1.2))
    ax.set_title(row["stem"], fontsize=8)
    ax.axis("off")

handles = [Rectangle((0, 0), 1, 1, fc=COLORS[i], ec="none") for i in range(4)]
fig.legend(handles, CLASS_NAMES, loc="lower center", ncol=4, frameon=False, fontsize=9)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

---
## 5. Pra-pemrosesan Citra  ·  *(Proposal §3.4)*

Tiga tahap sesuai proposal:

| Tahap | Operasi | Alasan |
|---|---|---|
| 1 | **Resize** | mengurangi jumlah piksel → beban komputasi turun |
| 2 | **Grayscale** | fokus pada bentuk/tepi/intensitas; HOG bekerja pada intensitas |
| 3 | **Peningkatan kontras (CLAHE)** | mempertegas perbedaan intensitas objek vs latar |

> **Catatan metodologis.** **CLAHE** (*Contrast Limited Adaptive Histogram Equalization*)
> digunakan karena bersifat **adaptif per wilayah** dan **membatasi amplifikasi noise**
> melalui *clip limit*. Hal ini penting untuk plat nomor: karakter plat adalah pola
> terang-gelap berfrekuensi tinggi, sehingga peningkatan kontras yang terlalu agresif
> justru akan menguatkan noise. Untuk plat, nilai `clipLimit` dibuat lebih rendah
> (1.5) dibanding jalur helm (2.0) agar tepi karakter tidak "pecah".

In [ ]:
CLAHE_CLIP = 1.5
CLAHE_GRID = (8, 8)
_clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID)

def to_grayscale(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def enhance_contrast(gray):
    return _clahe.apply(gray)

def preprocess_gray(img):
    """Pra-pemrosesan untuk jalur plat: grayscale -> CLAHE."""
    return enhance_contrast(to_grayscale(img))

print("Fungsi pra-pemrosesan siap. CLAHE clipLimit =", CLAHE_CLIP)

In [ ]:
# Demonstrasi pra-pemrosesan pada RoI plat
row = df[df["n_number plate"] > 0].iloc[0]
img = imread_robust(row["image_path"])
H, W = img.shape[:2]
plate = [b for b in row["boxes"] if b["cls"] == 3][0]
pb = box_tuple(plate)
X1, Y1 = int(pb[0]*W), int(pb[1]*H)
X2, Y2 = int(pb[2]*W), int(pb[3]*H)
crop = img[Y1:Y2, X1:X2]

gray = to_grayscale(crop)
eq = enhance_contrast(gray)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
axes[0].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"1. Crop plat (asli)\n{crop.shape[1]}×{crop.shape[0]}")
axes[1].imshow(gray, cmap="gray")
axes[1].set_title("2. Grayscale")
axes[2].imshow(eq, cmap="gray")
axes[2].set_title("3. CLAHE")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 2.6))
axes[0].hist(gray.ravel(), bins=64, color="#457b9d")
axes[0].set_title("Histogram grayscale")
axes[1].hist(eq.ravel(), bins=64, color="#2a9d8f")
axes[1].set_title("Histogram setelah CLAHE")
for ax in axes:
    ax.set_xlabel("Intensitas"); ax.set_ylabel("Frekuensi")
plt.tight_layout()
plt.show()

---
## 6. Penentuan Region of Interest (RoI)  ·  *(Proposal §3.5)*

Citra pada dataset memuat banyak area tidak relevan: jalan, latar belakang, dan bagian motor
yang bukan plat. RoI memfokuskan proses pada area yang mengandung plat sehingga:

- fitur yang diekstraksi lebih merepresentasikan objek,
- **jumlah kandidat yang harus dievaluasi turun drastis** → pengolahan lebih efisien.

### Definisi RoI Plat

Berdasarkan EDA §4, plat berada pada **45%–92% tinggi** *bounding box* pengendara, dengan
posisi horizontal 10%–95% lebarnya. RoI didefinisikan sebagai pita tersebut.

```
┌──────────────────────┐  ← y = 0%   (kepala/helm)
│                      │
├──────────────────────┤  ← y = 45%  ┬
│   ███ RoI PLAT ███   │             │ area pencarian plat
├──────────────────────┤  ← y = 92%  ┴
│                      │
└──────────────────────┘  ← y = 100%
```

Membatasi pencarian pada pita ini memangkas area yang harus dipindai menjadi sekitar
**47%** dari luas *bounding box* pengendara, dan jauh lebih kecil lagi dibanding seluruh citra.

> Ini adalah penerapan langsung temuan Hsu et al. (2015): **mempersempit area pencarian
> sebelum mencari plat mengurangi false positive dan mempercepat deteksi** (§2.6 proposal).

In [ ]:
ROI_Y_TOP    = 0.45
ROI_Y_BOTTOM = 0.92
ROI_X_LO     = 0.10
ROI_X_HI     = 0.95

def rider_roi_px(img_shape, rider_box,
                 y_top=ROI_Y_TOP, y_bottom=ROI_Y_BOTTOM,
                 x_lo=ROI_X_LO, x_hi=ROI_X_HI):
    """RoI plat dalam koordinat piksel dari box pengendara ternormalisasi."""
    H, W = img_shape[:2]
    x1, y1, x2, y2 = rider_box
    w, h = x2 - x1, y2 - y1
    X1 = int(np.clip((x1 + x_lo * w) * W, 0, W - 1))
    X2 = int(np.clip((x1 + x_hi * w) * W, 0, W - 1))
    Y1 = int(np.clip((y1 + y_top * h) * H, 0, H - 1))
    Y2 = int(np.clip((y1 + y_bottom * h) * H, 0, H - 1))
    return X1, Y1, X2, Y2

def crop_px(img, box):
    X1, Y1, X2, Y2 = box
    if X2 - X1 < 8 or Y2 - Y1 < 8:
        return None
    return img[Y1:Y2, X1:X2]

# Demonstrasi RoI
row = df[(df["n_rider"] > 0) & (df["n_number plate"] > 0)].iloc[2]
img = imread_robust(row["image_path"])
H, W = img.shape[:2]
rider = [b for b in row["boxes"] if b["cls"] == 2][0]
roi = rider_roi_px(img.shape, box_tuple(rider))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[0].add_patch(Rectangle((roi[0], roi[1]), roi[2]-roi[0], roi[3]-roi[1],
                            fill=False, ec="crimson", lw=2))
axes[0].set_title("Citra + RoI plat")
axes[1].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
for b in row["boxes"]:
    if b["cls"] == 3:
        axes[1].add_patch(Rectangle((b["x1"]*W, b["y1"]*H), (b["x2"]-b["x1"])*W,
                                    (b["y2"]-b["y1"])*H, fill=False, ec="#e9c46a", lw=2))
axes[1].set_title("Ground truth plat")
axes[2].imshow(cv2.cvtColor(crop_px(img, roi), cv2.COLOR_BGR2RGB))
axes[2].set_title(f"Crop RoI  {roi[2]-roi[0]}×{roi[3]-roi[1]}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

roi_area = (roi[2]-roi[0]) * (roi[3]-roi[1])
print(f"Luas RoI / luas citra : {roi_area/(W*H):.1%}")

---
## 7. Pembentukan Kandidat dengan Sliding Window Multi-Skala

Di dalam RoI, sistem perlu menemukan **calon lokasi plat** (*candidate proposals*) yang kemudian
dinilai oleh classifier. Pendekatan yang dipakai adalah **sliding window multi-skala** —
pendekatan klasik yang disebut pada proposal §2.6 sebagai cara kerja *sliding window* pada
visi komputer konvensional.

### Mengapa multi-skala?

Ukuran plat bervariasi (EDA §4: lebar plat 67–175 px, median 106 px). Karena itu dipakai
beberapa skala relatif terhadap lebar RoI:

| Parameter | Nilai | Alasan |
|---|---|---|
| Skala lebar | 20%, 28%, 36%, 44% dari lebar RoI | mencakup variasi ukuran plat |
| Rasio aspek | 2.1 (median hasil EDA) | bentuk plat |
| Langkah (*stride*) | 50% dari ukuran window | kompromi antara cakupan dan jumlah kandidat |

Window yang sangat kecil dibuang (lebar < 24 px) karena plat sekecil itu tidak dapat
dideskripsikan dengan andal oleh HOG berukuran 96×48.

In [ ]:
WIN_SIZE     = (96, 48)          # (lebar, tinggi) input HOG untuk plat
AR_WINDOW    = 2.1               # rasio aspek window
SCALES       = (0.20, 0.28, 0.36, 0.44)
STRIDE       = 0.50
MIN_WIN_W    = 24

HOG_ORIENT   = 9
HOG_PPC      = (8, 8)
HOG_CPB      = (2, 2)
HOG_NORM     = "L2-Hys"

def sliding_windows(X1, Y1, X2, Y2,
                    scales=SCALES, stride=STRIDE, ar=AR_WINDOW, min_w=MIN_WIN_W):
    """Hasilkan daftar window (x1,y1,x2,y2) di dalam RoI pada beberapa skala."""
    RW, RH = X2 - X1, Y2 - Y1
    out = []
    for sc in scales:
        w = max(min_w, int(RW * sc))
        h = max(12, int(round(w / ar)))
        if w > RW or h > RH:
            continue
        sx = max(6, int(w * stride))
        sy = max(4, int(h * stride))
        for yy in range(Y1, Y2 - h + 1, sy):
            for xx in range(X1, X2 - w + 1, sx):
                out.append((xx, yy, xx + w, yy + h))
    return out

def count_windows_for_row(row):
    img = imread_robust(row["image_path"])
    if img is None:
        return 0, 0
    n = 0
    for r in [b for b in row["boxes"] if b["cls"] == 2]:
        X1, Y1, X2, Y2 = rider_roi_px(img.shape, box_tuple(r))
        n += len(sliding_windows(X1, Y1, X2, Y2))
    return n, img.shape[0] * img.shape[1]

counts = [count_windows_for_row(r)[0] for _, r in df.iterrows()]
print(f"Rata-rata kandidat window per citra : {np.mean(counts):.0f}")
print(f"Median                              : {np.median(counts):.0f}")
print(f"Maksimum                            : {np.max(counts)}")
print(f"\nSebagai perbandingan, sliding window pada SELURUH citra dengan 4 skala")
print("yang sama akan menghasilkan ribuan kandidat per citra — inilah alasan RoI penting.")

In [ ]:
# Visualisasi kandidat window di dalam RoI
row = df[(df["n_rider"] > 0) & (df["n_number plate"] > 0)].iloc[1]
img = imread_robust(row["image_path"])
H, W = img.shape[:2]
rider = [b for b in row["boxes"] if b["cls"] == 2][0]
roi = rider_roi_px(img.shape, box_tuple(rider))
wins = sliding_windows(*roi)

vis = img.copy()
for (a1, b1, a2, b2) in wins:
    cv2.rectangle(vis, (a1, b1), (a2, b2), (80, 200, 80), 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(cv2.cvtColor(crop_px(img, roi), cv2.COLOR_BGR2RGB))
axes[0].set_title(f"RoI plat ({roi[2]-roi[0]}×{roi[3]-roi[1]})")
axes[1].imshow(cv2.cvtColor(crop_px(vis, roi), cv2.COLOR_BGR2RGB))
axes[1].set_title(f"{len(wins)} kandidat window")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

---
## 8. Ekstraksi Fitur dengan HOG  ·  *(Proposal §3.6)*

Setiap kandidat window diseragamkan ke **96 × 48 piksel**, lalu dideskripsikan dengan HOG.

### Mengapa HOG Cocok untuk Plat Nomor?

Plat nomor memiliki **pola garis dan tepi yang sangat khas**: deretan karakter dengan
orientasi gradien dominan vertikal dan horizontal. HOG merangkum distribusi orientasi gradien
ini menjadi vektor numerik yang **invarian terhadap perubahan iluminasi** (karena normalisasi
blok) namun tetap sensitif terhadap struktur tepi — tepat untuk membedakan area plat dari
bodi motor yang permukaannya lebih halus atau berpola acak.

### Konfigurasi

| Parameter | Nilai |
|---|---|
| Window | 96 × 48 px (rasio 2:1, mendekati rasio plat) |
| `orientations` | 9 (0°–180°, *unsigned*) |
| `pixels_per_cell` | 8 × 8 |
| `cells_per_block` | 2 × 2 |
| `block_norm` | `L2-Hys` |

Dimensi vektor fitur: **1980**.

In [ ]:
def extract_hog(gray_patch, win=WIN_SIZE, visualize=False):
    """Ekstraksi vektor fitur HOG dari patch grayscale."""
    g = cv2.resize(gray_patch, win, interpolation=cv2.INTER_AREA)
    return hog(g,
               orientations=HOG_ORIENT,
               pixels_per_cell=HOG_PPC,
               cells_per_block=HOG_CPB,
               block_norm=HOG_NORM,
               visualize=visualize,
               feature_vector=True)

def patch_gray(img_gray, box):
    X1, Y1, X2, Y2 = box
    return img_gray[Y1:Y2, X1:X2]

_probe = extract_hog(np.zeros((48, 96), np.uint8))
print("Dimensi vektor fitur HOG:", _probe.shape[0])

In [ ]:
# Visualisasi HOG pada patch plat vs patch non-plat
row = df[(df["n_rider"] > 0) & (df["n_number plate"] > 0)].iloc[3]
img = imread_robust(row["image_path"])
H, W = img.shape[:2]
gray = preprocess_gray(img)

plate = [b for b in row["boxes"] if b["cls"] == 3][0]
pb = box_tuple(plate)
plate_box = (int(pb[0]*W), int(pb[1]*H), int(pb[2]*W), int(pb[3]*H))

# patch negatif: window acak di RoI yang tidak menimpa plat
rng = np.random.RandomState(SEED)
rider = [b for b in row["boxes"] if b["cls"] == 2][0]
roi = rider_roi_px(img.shape, box_tuple(rider))
neg_box = None
for cand in sliding_windows(*roi):
    cn = (cand[0]/W, cand[1]/H, cand[2]/W, cand[3]/H)
    if iou_norm(cn, pb) < 0.05:
        neg_box = cand
        break

fig, axes = plt.subplots(2, 2, figsize=(9, 5))
for i, (box, title) in enumerate([(plate_box, "Patch PLAT (positif)"),
                                 (neg_box, "Patch NON-PLAT (negatif)")]):
    patch = patch_gray(gray, box)
    if patch.size == 0:
        continue
    fd, himg = extract_hog(patch, visualize=True)
    axes[i, 0].imshow(patch, cmap="gray")
    axes[i, 0].set_title(title, fontsize=9)
    axes[i, 1].imshow(himg, cmap="magma")
    axes[i, 1].set_title("Peta gradien HOG", fontsize=9)
    for ax in axes[i]:
        ax.axis("off")
plt.tight_layout()
plt.show()

---
## 9. Konstruksi Dataset Klasifikasi Patch

Classifier SVM dilatih pada **level patch**, bukan level citra. Dua jenis sampel dibentuk:

| Jenis | Cara memperoleh | Label |
|---|---|---|
| **Positif** | Potong tepat pada *bounding box* plat *ground truth* | 1 |
| **Negatif** | Window acak di dalam RoI yang **tidak** menimpa plat (IoU < 0.20) | 0 |

### Mengapa banyak sampel negatif?

Rasio alami plat terhadap seluruh kandidat window sangat kecil (plat menempati sebagian kecil
RoI). Jika jumlah negatif dan positif seimbang, classifier akan terlalu sering menghasilkan
*false positive* saat inferensi karena belum pernah melihat cukup banyak variasi latar belakang.
Karena itu ditambahkan **14 sampel negatif per pengendara**, ditambah **hard-negative mining**
pada §10.

### Pencegahan Kebocoran Data

Split dilakukan **berdasarkan citra**: seluruh patch dari satu citra berada di satu split saja.
Jika tidak, patch positif dan negatif dari citra yang sama dapat muncul di latih dan uji
sekaligus — model akan menghafal latar belakang spesifik citra tersebut dan evaluasi menjadi
tidak valid.

In [ ]:
N_NEG_PER_RIDER = 14
NEG_IOU_MAX     = 0.20

def rider_plate_boxes(boxes):
    return [b for b in boxes if b["cls"] == 3]

def collect_base_patches(df_sub, rng):
    """Kumpulkan patch positif (plat) dan negatif (acak di RoI) dari sekumpulan citra."""
    X, y = [], []
    for _, row in df_sub.iterrows():
        img = imread_robust(row["image_path"])
        if img is None:
            continue
        H, W = img.shape[:2]
        gray = preprocess_gray(img)
        plates = rider_plate_boxes(row["boxes"])
        plate_tuples = [box_tuple(p) for p in plates]

        # --- positif
        for pb in plate_tuples:
            X1 = int(np.clip(pb[0]*W, 0, W-1)); Y1 = int(np.clip(pb[1]*H, 0, H-1))
            X2 = int(np.clip(pb[2]*W, 0, W-1)); Y2 = int(np.clip(pb[3]*H, 0, H-1))
            if X2 - X1 < 8 or Y2 - Y1 < 5:
                continue
            patch = gray[Y1:Y2, X1:X2]
            if patch.size == 0:
                continue
            X.append(extract_hog(patch)); y.append(1)

        # --- negatif (acak di dalam RoI, tidak menimpa plat)
        for r in [b for b in row["boxes"] if b["cls"] == 2]:
            roi = rider_roi_px(img.shape, box_tuple(r))
            cands = sliding_windows(*roi)
            rng.shuffle(cands)
            got = 0
            for (a1, b1, a2, b2) in cands:
                nb = (a1/W, b1/H, a2/W, b2/H)
                if any(iou_norm(nb, pt) > NEG_IOU_MAX for pt in plate_tuples):
                    continue
                patch = gray[b1:b2, a1:a2]
                if patch.size == 0:
                    continue
                X.append(extract_hog(patch)); y.append(0)
                got += 1
                if got >= N_NEG_PER_RIDER:
                    break
    return np.array(X), np.array(y)

def mine_hard_negatives(df_sub, clf, rng, score_thr=-0.3, n_per_rider=12):
    """Hard-negative mining: ambil window yang SALAH dinilai tinggi oleh model saat ini."""
    X = []
    for _, row in df_sub.iterrows():
        img = imread_robust(row["image_path"])
        if img is None:
            continue
        H, W = img.shape[:2]
        gray = preprocess_gray(img)
        plate_tuples = [box_tuple(p) for p in rider_plate_boxes(row["boxes"])]
        for r in [b for b in row["boxes"] if b["cls"] == 2]:
            roi = rider_roi_px(img.shape, box_tuple(r))
            cands = sliding_windows(*roi)
            if not cands:
                continue
            feats = np.array([extract_hog(gray[b1:b2, a1:a2]) for a1, b1, a2, b2 in cands])
            scores = clf.decision_function(feats)
            got = 0
            for i in np.argsort(-scores):
                a1, b1, a2, b2 = cands[i]
                nb = (a1/W, b1/H, a2/W, b2/H)
                if any(iou_norm(nb, pt) > 0.25 for pt in plate_tuples):
                    continue
                if scores[i] < score_thr:
                    break
                X.append(feats[i]); got += 1
                if got >= n_per_rider:
                    break
    return np.array(X) if X else np.empty((0, _probe.shape[0]))

print("Fungsi pembentuk dataset patch siap.")

In [ ]:
# Split berdasarkan citra
unique_images = sorted(df["stem"].unique())
train_imgs, test_imgs = train_test_split(unique_images, test_size=0.30,
                                         random_state=SEED, shuffle=True)
train_set, test_set = set(train_imgs), set(test_imgs)
assert not (train_set & test_set)

df_train = df[df["stem"].isin(train_set)].reset_index(drop=True)
df_test  = df[df["stem"].isin(test_set)].reset_index(drop=True)

print(f"Citra latih : {len(df_train)}")
print(f"Citra uji   : {len(df_test)}")
print("Verifikasi: tidak ada irisan citra antara data latih dan uji.")

rng = np.random.RandomState(SEED)
X_train, y_train = collect_base_patches(df_train, rng)
X_test,  y_test  = collect_base_patches(df_test,  np.random.RandomState(SEED + 1))

print(f"\nPatch latih : {X_train.shape}  positif={int(y_train.sum())}  negatif={int((y_train==0).sum())}")
print(f"Patch uji   : {X_test.shape}  positif={int(y_test.sum())}  negatif={int((y_test==0).sum())}")

---
## 10. Klasifikasi dengan SVM  ·  

### 10.1 Prinsip Kerja

SVM mencari **hyperplane** yang memaksimalkan margin antar kelas:

$$\min_{w,b} \tfrac{1}{2}\lVert w \rVert^2 \quad \text{s.t.} \quad y_i (w^\top x_i + b) \ge 1$$

Dengan **soft margin** untuk data yang tidak sepenuhnya terpisah:

$$\min_{w,b,\xi} \tfrac{1}{2}\lVert w \rVert^2 + C \sum_i \xi_i$$

### 10.2 Hard-Negative Mining

Setelah model awal dilatih, model dijalankan pada data **latih** untuk menemukan window
yang **salah diberi skor tinggi** (false positive potensial). Window tersebut ditambahkan
sebagai sampel negatif, lalu model dilatih ulang. Proses ini diulang beberapa putaran.

Tujuan: mempersempit *decision boundary* pada area yang paling membingungkan model —
misalnya bodi motor yang memiliki tepi mirip karakter plat.

### 10.3 Pilihan Kernel

**Kernel linear** sesuai proposal §3.7: efisien untuk fitur HOG berdimensi tinggi (1980 dimensi)
dan memberikan keseimbangan akurasi–kecepatan yang terbaik menurut Hsu et al. (2015).
Standardisasi (`StandardScaler`) di-*fit* **hanya pada data latih**.

In [ ]:
HARD_ROUNDS = 2
HARD_THR    = -0.3

X_cur, y_cur = X_train.copy(), y_train.copy()
model = make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0, random_state=SEED))
model.fit(X_cur, y_cur)

history = []
for rnd in range(HARD_ROUNDS):
    X_hard = mine_hard_negatives(df_train, model, np.random.RandomState(SEED + 10 + rnd),
                                 score_thr=HARD_THR)
    if len(X_hard) == 0:
        print(f"Putaran {rnd+1}: tidak ada hard negative baru, berhenti.")
        break
    X_cur = np.vstack([X_cur, X_hard])
    y_cur = np.concatenate([y_cur, np.zeros(len(X_hard), dtype=int)])
    model = make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0, random_state=SEED))
    model.fit(X_cur, y_cur)
    acc = model.score(X_test, y_test)
    history.append({"putaran": rnd + 1, "n_hard": len(X_hard),
                    "total_patch": len(y_cur), "acc_uji": acc})
    print(f"Putaran {rnd+1}: +{len(X_hard)} hard negative  "
          f"(total {len(y_cur)} patch)  acc uji = {acc:.4f}")

if history:
    display(pd.DataFrame(history).round(4))
print(f"\nTotal patch latih akhir: {len(y_cur)}")

In [ ]:
# ---------------------------------------------------------------------------
# Ambang keputusan dikalibrasi HANYA pada data latih (out-of-fold), bukan pada
# data uji. Memilih ambang langsung dari data uji akan membuat metrik evaluasi
# bias optimistis karena ambang "mengetahui" jawaban uji.
# ---------------------------------------------------------------------------
from sklearn.model_selection import StratifiedKFold, cross_val_predict

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_scores = cross_val_predict(
    make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0, random_state=SEED)),
    X_cur, y_cur, cv=skf, method="decision_function")

prec_curve, rec_curve, thr_curve = precision_recall_curve(y_cur, oof_scores)
f1_curve = 2 * prec_curve * rec_curve / (prec_curve + rec_curve + 1e-12)
best_idx = int(np.nanargmax(f1_curve[:-1])) if len(thr_curve) else 0
DECISION_THR = float(thr_curve[best_idx]) if len(thr_curve) else 0.0

print(f"Ambang terpilih dari data LATIH (out-of-fold F1 maksimum): {DECISION_THR:.4f}")
print(f"  Precision (latih, OOF) : {prec_curve[best_idx]:.4f}")
print(f"  Recall    (latih, OOF) : {rec_curve[best_idx]:.4f}")

# Baru setelah ambang ditetapkan, terapkan ke data uji
scores_test = model.decision_function(X_test)
y_pred = (scores_test > DECISION_THR).astype(int)

# Kurva PR pada data uji hanya untuk pelaporan/visualisasi, bukan pemilihan ambang
prec_curve, rec_curve, thr_curve = precision_recall_curve(y_test, scores_test)
print(f"\nAverage Precision (AP) pada data uji: {average_precision_score(y_test, scores_test):.4f}")